# Data Loading and Raw EEG

## Overview

In this notebook, you will inspect an XDF recording and build an MNE `Raw` object. Most code cells are intentionally incomplete. Read the markdown, replace each `...`, and use the comments as hints. Do not continue until you can explain what the current cell produces.

## Learning objectives

- Explain what an XDF container stores.
- Use metadata and sample properties to identify EEG and marker streams.
- Extract timestamps, sampling rate, labels, and signal values.
- Convert time-by-channel data into an MNE `RawArray`.
- Save an annotated intermediate file.

## Background: What is XDF?

XDF stores multiple time-stamped streams in one file. LabRecorder may capture EEG, PsychoPy/LSL markers, and unrelated streams. Every stream has its own samples, timestamps, and metadata. Stream order is not guaranteed, so selecting stream 0 without inspecting it is unsafe.

In [7]:
# Setup cell: run this as provided.
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import pyxdf

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.eeg_utils import get_stream_channel_names, summarize_xdf_streams

mne.set_log_level("WARNING")

## Step 1: Load the XDF file

1. Confirm that the placeholder path points to your local, de-identified recording.
2. Read the documentation for `pyxdf.load_xdf`.
3. Call it and unpack its two return values into `streams` and `file_header`.
4. Print the number of streams.

**Check:** `streams` should be a list of stream dictionaries.

In [15]:
XDF_PATH = Path("../data/subSid_ses2_.xdf")

# Check that XDF_PATH exists and give a helpful error.
if not XDF_PATH.exists():
    raise FileNotFoundError(f"Add a de-identified XDF file at {XDF_PATH} or edit XDF_PATH.")



# TODO: load the file and unpack both returned objects.
streams, file_header = pyxdf.load_xdf(str(XDF_PATH))

# TODO: print how many streams were loaded.
len(streams)

2

## Step 2: Inspect available streams

An EEG stream usually has many samples, several channels, and a nonzero nominal sampling rate. A marker stream often has one channel, relatively few samples, and a nominal rate of 0 Hz. These are clues, not rules.

Use `summarize_xdf_streams(streams)` to make a table. Then write a loop that prints each stream's index, name, type, channel count, sampling rate, and sample count. Inspect the table before moving on.

In [21]:


# TODO: iterate over rows and print the requested fields.
# Hint: DataFrame.itertuples(index=False) is convenient.
stream_summary = summarize_xdf_streams(streams)
display(stream_summary)

for row in stream_summary.itertuples(index=False):

    for row in stream_summary.itertuples(index=False):
        print(
            f"name={row.name}, type={row.type}, "
            f"channel_count={row.channel_count}, "
            f"nominal_srate_hz={row.nominal_srate_hz}, "
            f"sample_count={row.sample_count}"
        )

,index,name,type,channel_count,nominal_srate_hz,sample_count
0,0,PsychoPyMarkers,Markers,1,0.0,76
1,1,WS-default,EEG,24,300.0,211648


name=PsychoPyMarkers, type=Markers, channel_count=1, nominal_srate_hz=0.0, sample_count=76
name=WS-default, type=EEG, channel_count=24, nominal_srate_hz=300.0, sample_count=211648
name=PsychoPyMarkers, type=Markers, channel_count=1, nominal_srate_hz=0.0, sample_count=76
name=WS-default, type=EEG, channel_count=24, nominal_srate_hz=300.0, sample_count=211648


## Step 3: Identify EEG and marker streams

Write down the evidence for your choices before entering the indices. Then select the two dictionaries from `streams`. Print the first 20 marker values and count all markers. If the values look like EEG voltages, you chose the wrong stream.

In [22]:
# STUDENT EDIT: obtain these indices from your stream table.
EEG_STREAM_INDEX = 1
MARKER_STREAM_INDEX = 0

# TODO: select the EEG and marker stream dictionaries.
eeg_stream = streams[EEG_STREAM_INDEX]
marker_stream =  streams[MARKER_STREAM_INDEX]


# TODO: inspect marker_stream['time_series'].
marker_samples = marker_stream['time_series']
print("Marker count:", len(marker_samples))
print("First 20 markers:", marker_samples[0:20])

Marker count: 76
First 20 markers: [['eyes_open_start'], ['eyes_open_end'], ['eyes_closed_start'], ['eyes_closed_end'], ['eyes_open_start'], ['eyes_open_end'], ['eyes_closed_start'], ['eyes_closed_end'], ['eyes_open_start'], ['eyes_open_end'], ['eyes_closed_start'], ['eyes_closed_end'], ['eyes_open_start'], ['eyes_open_end'], ['eyes_closed_start'], ['eyes_closed_end'], ['eyes_open_start'], ['eyes_open_end'], ['eyes_closed_start'], ['eyes_closed_end']]


## Step 4: Extract EEG data, timestamps, sampling rate, and channel names

XDF EEG samples are normally arranged as samples × channels. Extract `time_series` and `time_stamps`, convert both to NumPy arrays, and read `nominal_srate` from the stream's `info` dictionary. XDF metadata values are often stored inside one-item lists.

Use `get_stream_channel_names`. If its length does not equal the number of data columns, generate neutral names such as `EEG001`. Print the array shape, sampling rate, names, and duration.

In [23]:
# TODO: extract samples and timestamps as floating-point arrays.
eeg_samples = np.asarray(eeg_stream["time_series"], dtype=float)
eeg_timestamps = np.asarray(eeg_stream["time_stamps"], dtype=float)

# TODO: extract the nominal sampling rate.
sampling_rate = float(eeg_stream["info"]["nominal_srate"][0])

# Optional challenge: if nominal_srate is 0, infer the rate from timestamp differences.
# Hint: 1 / np.median(np.diff(eeg_timestamps))

# TODO: extract exactly one name per data column.
channel_names = get_stream_channel_names(eeg_stream)
if len(channel_names) != eeg_samples.shape[1]:
    channel_names = [f"EEG{i+1:03d}" for i in range(eeg_samples.shape[1])]

# TODO: print eeg sample shape, sampling rate, channel names, and recording duration.


duration = eeg_timestamps[-1] - eeg_timestamps[0]

print("EEG sample shape:", eeg_samples.shape)
print("Sampling rate:", sampling_rate, "Hz")
print("Channel names:", channel_names)
print("Recording duration:", duration, "seconds")

EEG sample shape: (211648, 24)
Sampling rate: 300.0 Hz
Channel names: ['P3', 'C3', 'F3', 'Fz', 'F4', 'C4', 'P4', 'Cz', 'Pz', 'Fp1', 'Fp2', 'T3', 'T5', 'O1', 'O2', 'X3', 'X2', 'F7', 'F8', 'X1', 'A2', 'T6', 'T4', 'TRG']
Recording duration: 705.4981554245514 seconds


In [ ]:
git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

## Step 5: Create an MNE Raw object

MNE expects channels × time and volts. Determine the XDF unit from metadata or device documentation. Use a scale of `1e-6` only if values are microvolts; use `1.0` if they are already volts.

Create `mne.Info` with `mne.create_info`, transpose and scale the data, then pass both to `mne.io.RawArray`. Finally, convert marker timestamps to seconds relative to the first EEG timestamp and attach them as `mne.Annotations`.

In [26]:
# STUDENT EDIT after checking the recording units.
EEG_SCALE_TO_VOLTS = None

# TODO: create the MNE Info object.
info = mne.create_info(
    ch_names=channel_names,
    sfreq=sampling_rate,
    ch_types= "eeg",
)

# TODO: scale to volts, transpose to channels x time, and create RawArray.
# STUDENT EDIT after checking the recording units.
EEG_SCALE_TO_VOLTS = 1e-6

# TODO: create the MNE Info object.
info = mne.create_info(
    ch_names=channel_names,
    sfreq=sampling_rate,
    ch_types="eeg",
)

# TODO: scale to volts, transpose to channels x time, and create RawArray.
eeg_data_volts = eeg_samples.T * EEG_SCALE_TO_VOLTS
raw = mne.io.RawArray(eeg_data_volts, info)

# TODO: flatten marker values into strings and align onsets to EEG time zero.
marker_timestamps = np.asarray(marker_stream["time_stamps"])
marker_descriptions = [str(m[0]) for m in marker_stream["time_series"]]

marker_onsets = marker_timestamps - eeg_stream["time_stamps"][0]
annotations = mne.Annotations(
    onset=marker_onsets, duration=np.zeros(len(marker_onsets)), description=marker_descriptions
)
raw.set_annotations(annotations)
raw
raw = mne.io.RawArray(eeg_data_volts, info)

# TODO: flatten marker values into strings and align onsets to EEG time zero.
marker_timestamps = np.asarray(marker_stream["time_stamps"])
marker_descriptions =  [str(m[0]) for m in marker_stream["time_series"]]

marker_onsets =  marker_timestamps - eeg_stream["time_stamps"][0]
annotations = mne.Annotations(
    onset=marker_onsets, duration=np.zeros(len(marker_onsets)), description=marker_descriptions
)
raw.set_annotations(annotations)
raw

TypeError: __init__() missing 1 required positional argument: 'info'

## Step 6: Plot the raw EEG

Use `raw.plot` to display about 10 seconds and no more than 20 channels. Start with automatic scaling. Scroll through multiple parts of the recording and verify that annotation labels appear at plausible times. Also calculate each channel's peak-to-peak amplitude in microvolts as a unit sanity check.

In [ ]:
# TODO: create the interactive raw-data plot.
raw.plot(
    duration=...,
    n_channels=...,
    scalings=...,
)

# TODO: compute peak-to-peak amplitude along the time axis and convert V to µV.
peak_to_peak_uv = ...
pd.Series(peak_to_peak_uv, index=raw.ch_names).describe()

## Step 7: Save the Raw object for later notebooks

Create the output directory if necessary and save with `raw.save`. Use `overwrite=True` only when you intentionally want to replace an earlier result. Confirm the absolute path after saving.

In [ ]:
OUTPUT_PATH = Path("../outputs/raw_eeg.fif")

# TODO: create the parent directory, save raw, and print the resolved path.
...

## Student exercises

1. Compare the nominal rate with a rate inferred from timestamps.
2. Build a marker-count table with `pandas.Series.value_counts`.
3. Add checks that reject an empty stream or mismatched sample/timestamp counts.

## Reflection questions

- What stream contained EEG, and what evidence supported your choice?
- How many markers were recorded? Was that consistent with 40 trials?
- Were channel names and units present in metadata?
- Why must marker times be aligned to the first EEG sample rather than the first marker?

## Summary

After completing the TODOs, you will have inspected an unknown XDF layout, selected streams using evidence, converted EEG to MNE's expected shape and units, aligned markers, and saved an annotated Raw file.